# 02 – Exploratory Data Analysis (EDA)

This notebook explores the CKD dataset through visualisations covering class distribution, feature distributions, correlation analysis, and outlier detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('tab10')

# ── Load ───────────────────────────────────────────────────────────────────
train_df = pd.read_csv('../dataset/Training_CKD_dataset.csv')
test_df  = pd.read_csv('../dataset/Testing_CKD_dataset.csv')

# ── Encode binary columns for correlation analysis ────────────────────────
BINARY_COLS = ['Diabetes', 'Hypertension', 'Smoking_Status', 'Family_History_Kidney']
df = train_df.copy()
for col in BINARY_COLS:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

print('Dataset loaded. Shape:', df.shape)

## 1. Class (Target) Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('tab10')

# ── Load ───────────────────────────────────────────────────────────────────
train_df = pd.read_csv('../dataset/Training_CKD_dataset.csv')
test_df  = pd.read_csv('../dataset/Testing_CKD_dataset.csv')

# ── Encode binary columns for correlation analysis ────────────────────────
BINARY_COLS = ['Diabetes', 'Hypertension', 'Smoking_Status', 'Family_History_Kidney']
df = train_df.copy()
for col in BINARY_COLS:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

print('Dataset loaded. Shape:', df.shape)

In [ ]:
stage_colors = {
    'Healthy Kidney':         '#2ecc71',
    'Mild CKD (Stage 1–2)':  '#f1c40f',
    'Moderate CKD (Stage 3)': '#e67e22',
    'Severe CKD (Stage 4)':   '#e74c3c',
    'Kidney Failure (Stage 5)':'#8e44ad',
}

class_counts = df['Target'].value_counts()
colors       = [stage_colors.get(lbl, '#3498db') for lbl in class_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(range(len(class_counts)), class_counts.values, color=colors, edgecolor='white', width=0.6)
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels([lbl.replace(' (', '\n(') for lbl in class_counts.index], fontsize=8)
axes[0].set_ylabel('Count')
axes[0].set_title('CKD Stage Distribution (Training Set)', fontweight='bold')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}', ha='center', fontsize=9, fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    class_counts.values, labels=None, colors=colors,
    autopct='%1.1f%%', startangle=140, pctdistance=0.75
)
for at in autotexts: at.set_fontsize(9)
axes[1].legend(wedges, class_counts.index, loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=8)
axes[1].set_title('Class Proportions', fontweight='bold')

plt.tight_layout()
plt.show()

print('\nClass counts:')
print(class_counts.to_string())

## 2. Key Biomarker Distributions

In [ ]:
key_features = ['eGFR', 'Serum_Creatinine', 'Blood_Urea_Nitrogen', 'Urine_Albumin',
                'Hemoglobin', 'Urine_Protein', 'Albumin_Creatinine_Ratio', 'Potassium']

targets = sorted(df['Target'].unique())
pal     = [stage_colors.get(t, '#3498db') for t in targets]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    for i, target in enumerate(targets):
        subset = df[df['Target'] == target][feat].dropna()
        ax.hist(subset, bins=30, alpha=0.55, label=target, color=pal[i], edgecolor='none')
    ax.set_title(feat, fontweight='bold', fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(labelsize=8)

handles = [mpatches.Patch(color=c, label=t) for t, c in zip(targets, pal)]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Key Biomarker Distributions by CKD Stage', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## 3. Box Plots – Spread & Outliers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

box_features = ['eGFR', 'Serum_Creatinine', 'Urine_Albumin', 'Hemoglobin', 'Potassium', 'Blood_Urea_Nitrogen']

for ax, feat in zip(axes, box_features):
    data = [df[df['Target'] == t][feat].dropna().values for t in targets]
    bp   = ax.boxplot(data, patch_artist=True, notch=False, widths=0.5)
    for patch, color in zip(bp['boxes'], pal):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels([t.split('(')[0].strip() for t in targets], fontsize=7, rotation=15, ha='right')
    ax.set_title(feat, fontweight='bold', fontsize=9)

fig.suptitle('Box Plots – Feature Spread Across CKD Stages', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_corr = df.copy()
le = LabelEncoder()
df_corr['Target'] = le.fit_transform(df_corr['Target'])

corr = df_corr.select_dtypes(include=[np.number]).corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0,
            linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Top Features Correlated with Target

In [ ]:
target_corr = corr['Target'].drop('Target').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
colors_bar = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(target_corr[:15])))
bars = ax.barh(target_corr[:15].index[::-1], target_corr[:15].values[::-1],
               color=colors_bar, edgecolor='none')
ax.set_xlabel('|Correlation with Target|')
ax.set_title('Top 15 Features Correlated with CKD Stage', fontweight='bold')
for bar, val in zip(bars, target_corr[:15].values[::-1]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print('Top 10 features correlated with Target:')
print(target_corr[:10].to_string())

## 6. Risk Factor Analysis (Categorical)

In [ ]:
risk_factors = ['Diabetes', 'Hypertension', 'Smoking_Status', 'Family_History_Kidney']
BINARY_COLS2 = risk_factors

df_risk = train_df.copy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, risk_factors):
    ct = pd.crosstab(df_risk[col], df_risk['Target'], normalize='index')
    ct.plot(kind='bar', ax=ax, stacked=True,
            color=[stage_colors.get(c, '#3498db') for c in ct.columns],
            edgecolor='none', legend=False)
    ax.set_title(col.replace('_', ' '), fontweight='bold', fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=0)

handles = [mpatches.Patch(color=stage_colors.get(t, '#3498db'), label=t) for t in targets]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=7, bbox_to_anchor=(0.5, -0.08))
fig.suptitle('CKD Stage Proportion by Risk Factor', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

## 7. Pair Plot – Key Features

In [ ]:
pair_feats   = ['eGFR', 'Serum_Creatinine', 'Blood_Urea_Nitrogen', 'Urine_Albumin', 'Target']
pair_df      = df[pair_feats].dropna().copy()

pair_pal = {t: c for t, c in zip(targets, pal)}
g = sns.pairplot(pair_df, hue='Target', palette=pair_pal, diag_kind='kde', plot_kws={'alpha': 0.4})
g.fig.suptitle('Pair Plot – Key Kidney Function Markers', y=1.02, fontsize=12, fontweight='bold')
plt.show()

## 8. EDA Summary

**Key Observations:**

1. **eGFR** is the strongest predictor of CKD stage — it decreases sharply with disease severity.
2. **Serum Creatinine** and **Blood Urea Nitrogen** rise steeply in Stages 4 and 5.
3. **Urine Albumin** and **ACR** are strongly elevated in kidney failure.
4. **Hemoglobin** drops progressively with CKD stage (anaemia of CKD).
5. **Diabetes** and **Hypertension** patients have a higher proportion of severe CKD stages.
6. The dataset is reasonably balanced across the five classes.